<a href="https://colab.research.google.com/github/shay-bagaria/resistancemap-za/blob/main/notebooks/ResistanceMap_ZA_04_Pharmacokinetic_Modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# CELL 1: Environment & Matrix Loading
# ==============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Initialising Pharmacokinetic Engine...")

# Ensure output directory for our visual models exists
os.makedirs("outputs/plots", exist_ok=True)
os.makedirs("outputs/models", exist_ok=True)

# Load the mutation matrix from Module B
mutation_csv = "data/mutation_tables/kzn_mutation_frequencies.csv"

try:
    df_mutations = pd.read_csv(mutation_csv)
    print(f"✅ Mutation matrix loaded. Found {len(df_mutations)} unique major mutations.")
except FileNotFoundError:
    print(f"⚠️ Warning: {mutation_csv} not found. Using simulated KZN baseline data for model testing.")
    # Fallback simulated data if the previous notebook was run on a different virtual session
    data = {'Mutation': ['K103N', 'M184V', 'V106M', 'Y181C'], 'Prevalence (%)': [45.2, 38.5, 12.1, 8.4]}
    df_mutations = pd.DataFrame(data)

In [ ]:
# ==============================================================================
# CELL 2: Pharmacokinetic Elimination Models
# ==============================================================================

# Define clinical PK parameters for common ARVs
# Format: [Half-life (hours), Steady State Cmax (mg/L), Minimum Inhibitory Concentration (MIC) (mg/L)]
pk_parameters = {
    "Efavirenz (EFV)": {"t_half": 45.0, "C_max": 4.0, "MIC": 1.0},
    "Tenofovir (TDF)": {"t_half": 17.0, "C_max": 0.3, "MIC": 0.05},
    "Lamivudine (3TC)": {"t_half": 5.0,  "C_max": 1.5, "MIC": 0.5}
}

def calculate_decay(c_max, t_half, time_hours):
    """Calculates drug concentration using one-compartment exponential decay."""
    k_e = np.log(2) / t_half
    return c_max * np.exp(-k_e * time_hours)

# Simulate a 14-day (336 hours) window following treatment cessation
time_array = np.arange(0, 336, 1)
decay_curves = pd.DataFrame({"Hours": time_array})

print("Calculating pharmacokinetic decay curves...")

for drug, params in pk_parameters.items():
    decay_curves[drug] = calculate_decay(params["C_max"], params["t_half"], time_array)

print("✅ Differential decay matrix constructed.")

In [ ]:
# ==============================================================================
# CELL 3: Visualising the Pharmacokinetic Tail
# ==============================================================================

sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 6))

colors = {"Efavirenz (EFV)": "crimson", "Tenofovir (TDF)": "steelblue", "Lamivudine (3TC)": "forestgreen"}

for drug in pk_parameters.keys():
    plt.plot(decay_curves["Hours"], decay_curves[drug], label=drug, color=colors[drug], linewidth=2.5)
    # Plot the Minimum Inhibitory Concentration (MIC) threshold
    plt.axhline(y=pk_parameters[drug]["MIC"], color=colors[drug], linestyle="--", alpha=0.5)

plt.fill_between(decay_curves["Hours"], pk_parameters["Efavirenz (EFV)"]["MIC"], decay_curves["Efavirenz (EFV)"],
                 where=(decay_curves["Efavirenz (EFV)"] < pk_parameters["Efavirenz (EFV)"]["MIC"]) &
                       (decay_curves["Efavirenz (EFV)"] > 0.1),
                 color='crimson', alpha=0.2, label="EFV Sub-Therapeutic Vulnerability Window")

plt.title("ARV Plasma Concentration Decay Following Treatment Cessation", fontsize=14, fontweight="bold")
plt.xlabel("Time Since Last Dose (Hours)", fontsize=12)
plt.ylabel("Plasma Concentration (mg/L) - Log Scale", fontsize=12)
plt.yscale("log")
plt.legend(loc="upper right")
plt.tight_layout()

# Save the plot for the GitHub repository and future research papers
plot_path = "outputs/plots/pk_decay_curves.png"
plt.savefig(plot_path, dpi=300)
plt.show()

print(f"✅ Pharmacokinetic visual model exported to {plot_path}")

In [ ]:
# ==============================================================================
# CELL 4: The Resistance Risk Index (RRI) Synthesis
# ==============================================================================

print("Computing final Resistance Risk Index (RRI) scores...\n")

# Calculate vulnerability window (hours spent between MIC and 10% of MIC)
for drug, params in pk_parameters.items():
    t_mic = -np.log(params["MIC"] / params["C_max"]) / (np.log(2) / params["t_half"])
    t_clear = -np.log((params["MIC"]*0.1) / params["C_max"]) / (np.log(2) / params["t_half"])
    pk_parameters[drug]["Vulnerability_Hours"] = max(0, t_clear - t_mic)

# Map mutations to their primary drug drivers
mutation_drug_map = {
    "K103N": "Efavirenz (EFV)",
    "V106M": "Efavirenz (EFV)",
    "M184V": "Lamivudine (3TC)",
    "K65R": "Tenofovir (TDF)"
}

rri_results = []

for index, row in df_mutations.iterrows():
    mut = row["Mutation"]
    prev = row["Prevalence (%)"]

    # Identify the drug driving this mutation
    driver_drug = mutation_drug_map.get(mut, "Unknown")
    if driver_drug in pk_parameters:
        vuln_hours = pk_parameters[driver_drug]["Vulnerability_Hours"]
        # RRI Formula: (Prevalence) x (Vulnerability Window in Days)
        rri_score = prev * (vuln_hours / 24)
        rri_results.append({"Mutation": mut, "Driver Drug": driver_drug,
                            "Prevalence (%)": prev, "Vulnerability (Days)": round(vuln_hours/24, 1),
                            "RRI Score": round(rri_score, 2)})

df_rri = pd.DataFrame(rri_results).sort_values(by="RRI Score", ascending=False)
df_rri.to_csv("outputs/models/kzn_rri_scores.csv", index=False)

print("🎯 RRI Computation Complete. Critical Targets Identified:")
print("-" * 75)
print(df_rri.to_string(index=False))
print("-" * 75)
print("Conclusion: ChromaTrace physical assay must be calibrated to detect the driver drug of the highest RRI Score.")